# Assets at risk

**Which repositories carry the backlog, which of them offer an attacker a foothold, and which
are falling behind?**

Prioritization to Prediction **volume 5**, whose unit of analysis is the asset rather than the
vulnerability: *"the fact that we manage vulnerabilities in assets rather than in a vacuum
requires us to know where risk isn't, where it is now, and where it will eventually be."* For a
code register the asset is the repository branch, and v5's asset categories — it compares
Windows against Linux against network appliances — are the language ecosystem here.

Set `scope` to `sca` or `sast`. On the `os` register every column on this page is empty, and
that is the honest answer rather than a broken one: `config.FETCH_ASSET_FIELDS` is off there, so
those findings have no asset to be counted against.

## How to read this notebook

Every cell answers one question and shows one thing. Run them in order the first time; after
that any cell can be re-run on its own.

**Set the widgets at the top before you run anything.** `catalog` has no default on purpose, and
`scope` decides which register this is. Set the notebook to **Run accessed commands** (the
dropdown beside *Run all*) if you want a widget change to re-run the cells that depend on it —
otherwise you will change the filter and read a chart drawn under the old one.

Everything here reads one scan, pinned in cell 1. Every figure is over the **high-risk**
population unless it says otherwise.

In [ ]:
import os, sys

_paths = []
try:
    _paths.append(dbutils.widgets.get("module_path"))
except Exception:  # noqa: BLE001 -- the widget does not exist yet on a first run
    pass
_here = os.getcwd()
_paths += [_here, os.path.dirname(_here)]
for _p in _paths:
    if _p and os.path.exists(os.path.join(_p, "panels.py")):
        sys.path.insert(0, _p)
        break
else:
    raise RuntimeError("brick modules are not on sys.path -- see brick/README.md, step 2")

import panels, figures, tiles

PAGE = {"weaknesses": ("15", None)}

panels.declare_widgets(**PAGE)
ctx = panels.context(spark, **{name: str(spec[0]) for name, spec in PAGE.items()})
displayHTML(tiles.scan_zone_from(panels.last_scan(spark, ctx).first()))

## The register, by ecosystem

Asset prevalence and vulnerability density together: how many repositories there are of each
kind, and how much each of them is carrying. Lower density is generally better, but higher is
not automatically worse — v5 is explicit that some assets carry more simply because of what they
are.

Read `window_months` and `assets_flowing` before the capacity columns. They are what stops a
confident-looking split over three repositories and one month passing for a trend.

In [ ]:
display(panels.asset_profile(spark, ctx))

## Vulnerability density (v5 Fig. 10)

The p25–p75 spread with the median marked, not a bar chart of medians. v5 reports three
percentiles because the distribution is far too skewed for one number to describe — its own
sample runs from under 10 findings per asset to over 1,000 — and the spread is the part worth
comparing.

Both populations are on the frame. An ecosystem whose total density is high but whose high-risk
density is not has a triage problem; one where the two are close has a supply-chain problem.

In [ ]:
_density = panels.asset_density(spark, ctx).where("population = 'high_risk'").toPandas()
figures.render(figures.density_range(_density))

## Where the footholds are (v5 Fig. 11)

v5's framing, and the reason this is a headline rather than a column: *"it's often said that
just one opening is needed to successfully compromise a system"*. 70% of Windows systems and 40%
of Linux systems cleared that bar in their sample.

**That number is not comparable to this one** — different population, different positive class.
See *Reading coverage and efficiency* in `brick/README.md`. The question is the same; the scale
is not.

Chart ▸ Bar · X: `asset_group` · Y: `assets_with_high_risk_pct` · Horizontal: on · X title: share of repositories with at least one open high-risk finding

In [ ]:
display(panels.asset_footholds(spark, ctx))

## Can each ecosystem keep up? (v5 Fig. 21)

The share of repositories falling behind, keeping up and gaining ground, over the same dead band
around zero the monthly capacity table uses — a one-finding swing must not flip a repository
between verdicts.

**Blank when the register has no scan log.** These are rates per *watched* month, and a register
that has not recorded when it started watching has none. Reconstructed capacity is the one thing
this pipeline refuses to headline, so the columns are empty rather than back-dated from the
API's own dates.

In [ ]:
_split = panels.asset_capacity(spark, ctx).toPandas()
figures.render(figures.capacity_split(_split))

In [ ]:
display(panels.asset_capacity(spark, ctx))

## Weakness classes (`sast` only)

What kinds of weakness the static-analysis register is made of, and how many of each the rule
calls high risk.

Two things about this table. A finding with two CWEs is counted under both, so **the counts do
not sum to the register** — it is a tally, not a partition. And `unclassified` is not a rounding
error: a weakness class the rule cannot decide on leaves both sides of coverage and efficiency,
and the published bounds beside those rates are what that population could do to them.

Empty on `sca` and `os`, which have no CWE.

In [ ]:
display(panels.weakness_mix(spark, ctx, ctx.int_param('weaknesses', 15)))

## How much of this is the rule?

The same sensitivity sweep `02_program_performance` shows, and it matters more here, not less:
on a `sast` register the high-risk label is a weakness class and a severity somebody typed, with
no exploitation data anywhere in the chain.

A subset cannot come out "wrong" — each is scored against itself — so read the shape of the
trade rather than picking a winner. `high_risk` and `unknown` are on every row precisely so that
a rule which buys efficiency by shrinking the positive class cannot hide it.

In [ ]:
display(panels.rule_sweep(spark, ctx))